In [64]:
!pip install spacy
!python -m spacy download es_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.0/568.0 MB 1.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import pandas as pd
import torch
#from TP2 import LogisticRegression

pd.options.display.float_format = (
    lambda x: '0' if x == 0 else f'{x}'
)

class TextProcessor:
    def __init__(self):
        self.nlp = spacy.load("es_core_news_lg")
        self.train_vectorizer = TfidfVectorizer(analyzer=self.preprocess_document)

    def preprocess_document(self, doc : str):
        preprocess_doc = self.nlp(doc.lower())
        keywords = [token.lemma_ for token in preprocess_doc if token.is_alpha and not token.is_stop]
        return keywords

    def show_descriptor(self, train_vectorised, tokens):
        df = pd.DataFrame(train_vectorised.toarray(), columns = tokens)
        print(df.round(5))

    def make_descriptor(self, train_vectorised):
        return torch.tensor(train_vectorised.toarray(), dtype=torch.float32)

    def tfidf(self, corpus : list[str]):
        train_vectorized = self.train_vectorizer.fit_transform(corpus)
        tokens = self.train_vectorizer.get_feature_names_out()
        return train_vectorized, tokens

In [79]:
data_test = [
    "¡Qué maravilloso es que nadie necesite esperar ni un solo momento antes de comenzar a mejorar el mundo!",
    "En un lugar de la Mancha, inmiscuyéndose en sustancias superfluas",
    "Nombras tú mi nombre Como jamás lo dijo un hombre Agapimú Tocas mi cintura Como la hiedra toca altura Agapimú",
    "La guerra es la paz. La libertad es la esclavitud. La ignorancia es la fuerza",
    "Tiemblas temblamos temblaremos temblaís tembló amor mío Como una gota de rocío Agapimú Entras en mi cuerpo Como la lluvia entra en mi huerto Agapimú",
    "Quien controla el pasado controla el futuro. Quien controla el presente controla el pasado",
    "El pasado"
]

def test_analyzer():
    text_processor = TextProcessor()
    for doc in data_test:
        print(text_processor.preprocess_document(doc))

    train_vectorized, tokens = text_processor.tfidf(data_test)
    text_processor.show_descriptor(train_vectorized, tokens)

test_analyzer()

['maravilloso', 'necesitar', 'esperar', 'momento', 'comenzar', 'mejorar', 'mundo']
['lugar', 'mancha', 'inmiscuir él', 'sustancia', 'superfluas']
['nombras', 'nombre', 'jamás', 'hombre', 'agapimú', 'tocar', 'cintura', 'hiedra', 'tocar', 'altura', 'agapimú']
['guerra', 'paz', 'libertad', 'esclavitud', 'ignorancia', 'fuerza']
['tiembla', 'temblar', 'temblaremos', 'temblaís', 'temblo', 'amor', 'gota', 'rocío', 'agapimú', 'entrar', 'cuerpo', 'lluvia', 'entrar', 'huerto', 'agapimú']
['controlar', 'controlar', 'futuro', 'controlar', 'presente', 'controlar']
[]
   agapimú  altura    amor  cintura  comenzar  controlar  cuerpo  entrar  \
0        0       0       0        0   0.37796          0       0       0   
1        0       0       0        0         0          0       0       0   
2  0.44761 0.26962       0  0.26962         0          0       0       0   
3        0       0       0        0         0          0       0       0   
4  0.39398       0 0.23732        0         0          0 0.

In [ ]:
class ReadDataset:
    def __init__(self, path : str):
        self.path = path

    def read_dataset(self):
        dataset = pd.read_excel(self.path, sheet_name="Sheet1")
        corpus = []
        labels = []
        for _,row in dataset.iterrows():
            corpus.append(row["Segment"])
            corpus.append(row["Proposal"])
            labels.append(1)
            labels.append(0)
        return corpus, labels

# class TrainModel:
#     def
dataset = ReadDataset("dataset.xlsx")
corpus, labels = dataset.read_dataset()
corpus_train, corpus_test, label_train, label_test = train_test_split(corpus, labels, test_size = 0.3)

In [6]:
#!pip install stanza
import stanza
stanza.download('es', package='ancora', processors='tokenize,mwt,pos,lemma', verbose=False)

class Processor:
    def __init__(self):
        self.nlp = stanza.Pipeline(processors='tokenize,mwt,pos,lemma', lang='es', use_gpu=True, package='ancora')
        self.train_vectorizer = TfidfVectorizer(analyzer=self.preprocess_document)

    def preprocess_document(self, doc : str):
        processed = self.nlp(doc)
        lemmas = []
        for sent in processed.sentences:
            for word in sent.words:
                if word.upos != "PUNCT":
                    lemmas.append(word.lemma.lower())
        return lemmas

    def show_descriptor(self, train_vectorised, tokens):
        df = pd.DataFrame(train_vectorised.toarray(), columns = tokens)
        print(df.round(5))

    def make_descriptor(self, train_vectorised):
        return torch.tensor(train_vectorised.toarray(), dtype=torch.float32)

    def tfidf(self, corpus : list[str]):
        train_vectorized = self.train_vectorizer.fit_transform(corpus)
        tokens = self.train_vectorizer.get_feature_names_out()
        return train_vectorized, tokens

data_test = [
    "¡Qué maravilloso es que nadie necesite esperar ni un solo momento antes de comenzar a mejorar el mundo!",
    "En un lugar de la Mancha, inmiscuyéndose en sustancias superfluas",
    "Nombras tú mi nombre Como jamás lo dijo un hombre Agapimú Tocas mi cintura Como la hiedra toca altura Agapimú",
    "La guerra es la paz. La libertad es la esclavitud. La ignorancia es la fuerza",
    "Tiemblas temblamos temblaremos temblaís tembló amor mío Como una gota de rocío Agapimú Entras en mi cuerpo Como la lluvia entra en mi huerto Agapimú",
    "Quien controla el pasado controla el futuro. Quien controla el presente controla el pasado",
    "El pasado"
]

processor = Processor()
train_vectorized, tokens = processor.tfidf(data_test)
print(tokens)

['a' 'agapimú' 'altura' 'amor' 'antes' 'cintura' 'comenzar' 'como'
 'controlar' 'cuerpo' 'de' 'decir' 'el' 'en' 'entrar' 'entras'
 'esclavitud' 'esperar' 'fuerza' 'futuro' 'gota' 'guerra' 'hiedra'
 'hombre' 'huerto' 'ignorancia' 'inmiscuyar' 'jamás' 'libertad' 'lluvia'
 'lugar' 'mancha' 'maravilloso' 'mejorar' 'mi' 'momento' 'mundo' 'mío'
 'nadie' 'necesitar' 'ni' 'nombrar' 'nombre' 'pasado' 'paz' 'presente'
 'que' 'quien' 'qué' 'rocío' 'ser' 'solo' 'superfluo' 'sustancia'
 'temblar' 'temblaí' 'tiembla' 'tocar' 'tocas' 'tú' 'uno' 'él']
